In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Messy_Employee_dataset.csv')

In [23]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        1020 non-null   str    
 1   First_Name         1020 non-null   str    
 2   Last_Name          1020 non-null   str    
 3   Age                809 non-null    float64
 4   Department_Region  1020 non-null   str    
 5   Status             1020 non-null   str    
 6   Join_Date          1020 non-null   str    
 7   Salary             996 non-null    float64
 8   Email              1020 non-null   str    
 9   Phone              1020 non-null   int64  
 10  Performance_Score  1020 non-null   str    
 11  Remote_Work        1020 non-null   bool   
dtypes: bool(1), float64(2), int64(1), str(8)
memory usage: 88.8 KB


In [25]:
df.isnull().sum()

Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64

In [26]:
df.duplicated().sum()

np.int64(0)

In [27]:
df['Phone'].describe()

count    1.020000e+03
mean    -4.942253e+09
std      2.817326e+09
min     -9.994973e+09
25%     -7.341992e+09
50%     -4.943997e+09
75%     -2.520391e+09
max     -3.896086e+06
Name: Phone, dtype: float64

In [28]:
print("=== DATA QUALITY REPORT (Before Cleaning) ===")
print(f"\nTotal rows: {df.shape[0]}, Total columns: {df.shape[1]}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nNull values per column:\n{df.isnull().sum()}")
print(f"\nData types:\n{df.dtypes}")

=== DATA QUALITY REPORT (Before Cleaning) ===

Total rows: 1020, Total columns: 12

Duplicate rows: 0

Null values per column:
Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64

Data types:
Employee_ID              str
First_Name               str
Last_Name                str
Age                  float64
Department_Region        str
Status                   str
Join_Date                str
Salary               float64
Email                    str
Phone                  int64
Performance_Score        str
Remote_Work             bool
dtype: object


In [ ]:
## Data Quality Report (Before Cleaning)

- Dataset size: 1,020 rows, 12 columns
- Duplicate rows: 0 (checked and confirmed none)
- Missing values: Age (211 missing, ~21%), Salary (24 missing, ~2%). All other 
columns fully populated.
- Data type issues: `Join_Date` is stored as text instead of a proper date type. 
`Phone` is stored as `int64` but contains only negative values — indicates data 
corruption, investigated below.

In [29]:
df['Age'].value_counts().head(10)

Age
40.0    210
25.0    206
30.0    205
35.0    188
Name: count, dtype: int64

In [30]:
age_median = df['Age'].median()
print(f"Median Age: {age_median}")

df['Age'] = df['Age'].fillna(age_median)

Median Age: 30.0


In [ ]:
Missing Data Handling — Age:

Investigated the Age column before imputing and found 
only 4 distinct values in the entire dataset (25, 30, 35, 40), each appearing roughly 
equally. This indicates Age was generated from fixed buckets rather than representing 
precise individual ages. Given this, missing values (211 rows) were filled using the 
median (30.0) rather than the mean, since median guarantees the fill value matches 
one of the dataset's existing discrete buckets rather than producing an artificial 
decimal that doesn't fit the data's pattern.

In [31]:
df['Salary'].describe()

count       996.000000
mean      85155.056396
std       19873.727918
min       50047.320000
25%       68392.487500
50%       85547.870000
75%      100974.027500
max      119971.650000
Name: Salary, dtype: float64

In [32]:
salary_median = df['Salary'].median()
print(f"Median Salary: {salary_median}")

df['Salary'] = df['Salary'].fillna(salary_median)

Median Salary: 85547.87


In [33]:
df.isnull().sum()

Employee_ID          0
First_Name           0
Last_Name            0
Age                  0
Department_Region    0
Status               0
Join_Date            0
Salary               0
Email                0
Phone                0
Performance_Score    0
Remote_Work          0
dtype: int64

In [ ]:
Missing Data Handling — Salary:

Unlike Age, Salary's mean ($85,155) and median 
($85,548) are very close, indicating a normal, unbucketed distribution with no major 
skew. Missing values (24 rows) were filled using the median ($85,547.87), kept 
consistent with the Age approach and generally more robust to outliers than the mean.

In [34]:
df['Join_Date'] = pd.to_datetime(df['Join_Date'])
df['Join_Date'].head()

0   2021-04-02
1   2020-07-10
2   2023-12-07
3   2021-11-27
4   2022-01-05
Name: Join_Date, dtype: datetime64[us]

In [ ]:
Data Type Correction — Join_Date: C

onverted from text (`str`) to proper `datetime` 
type using `pd.to_datetime()`. This enables chronological sorting, date filtering, and 
tenure calculations that weren't possible while stored as plain text.

In [35]:
df['Phone'] = df['Phone'].astype(str)
df['Phone'] = 'CORRUPTED_' + df['Phone']
df['Phone'].head()

0    CORRUPTED_-1651623197
1    CORRUPTED_-1898471390
2    CORRUPTED_-5596363211
3    CORRUPTED_-3476490784
4    CORRUPTED_-1586734256
Name: Phone, dtype: str

In [ ]:
Data Type Correction — Phone (Corrupted Data):

All 1,020 Phone values were found to 
be negative integers, ranging from roughly -3.9 million to -9.99 billion. This pattern 
indicates integer overflow — the original phone numbers were too large to fit a 
standard 32-bit integer type and "wrapped around" into negative values during data 
generation/import. Since the original numbers cannot be mathematically recovered from 
this corrupted state, the column was converted to string type and explicitly flagged 
with a `CORRUPTED_` prefix, rather than attempting to guess or fabricate replacement 
values. This preserves transparency about the data's limitations.

In [36]:
Q1 = df['Salary'].quantile(0.25)
Q3 = df['Salary'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")

outliers = df[(df['Salary'] < lower_bound) | (df['Salary'] > upper_bound)]
print(f"\nNumber of outliers: {len(outliers)}")
outliers

Q1: 68811.2325, Q3: 100372.6625, IQR: 31561.430000000008
Lower bound: 21469.087499999987, Upper bound: 147714.80750000002

Number of outliers: 0


,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work


In [37]:
Q1_age = df['Age'].quantile(0.25)
Q3_age = df['Age'].quantile(0.75)
IQR_age = Q3_age - Q1_age

lower_age = Q1_age - 1.5 * IQR_age
upper_age = Q3_age + 1.5 * IQR_age

age_outliers = df[(df['Age'] < lower_age) | (df['Age'] > upper_age)]
print(f"Age outliers: {len(age_outliers)}")

Age outliers: 0


In [ ]:
Outlier Detection (IQR Method): Applied the IQR method (1.5× rule) to both Salary 
and Age. 

- Salary: Bounds calculated at $21,469 (lower) and $147,715 (upper). Zero outliers 
found — all salary values fall within a reasonable, expected range.
- Age:** Zero outliers found, consistent with the earlier finding that Age is limited 
to only 4 discrete bucketed values (25, 30, 35, 40).

No values were capped, removed, or altered as a result of this check — the data was 
already within normal statistical bounds.

In [38]:
print("Status values:", df['Status'].unique())
print("\nPerformance_Score values:", df['Performance_Score'].unique())
print("\nDepartment_Region values:", df['Department_Region'].unique())

Status values: <StringArray>
['Active', 'Pending', 'Inactive']
Length: 3, dtype: str

Performance_Score values: <StringArray>
['Average', 'Excellent', 'Good', 'Poor']
Length: 4, dtype: str

Department_Region values: <StringArray>
[    'DevOps-California',         'Finance-Texas',          'Admin-Nevada',
    'Cloud Tech-Florida',           'Sales-Texas',       'DevOps-New York',
   'Cloud Tech-New York',           'HR-New York',    'Finance-California',
         'Sales-Florida',       'DevOps-Illinois',     'Cloud Tech-Nevada',
        'Admin-Illinois',   'Cloud Tech-Illinois',        'Sales-Illinois',
        'Finance-Nevada',           'Admin-Texas',        'Sales-New York',
       'Finance-Florida',      'Admin-California',              'HR-Texas',
         'HR-California',             'HR-Nevada',      'Cloud Tech-Texas',
      'Sales-California',           'HR-Illinois',        'Admin-New York',
         'Admin-Florida',            'HR-Florida',          'Sales-Nevada',
      'Fin

In [ ]:
Standardisation Check: 

Inspected `Status`, `Performance_Score`, and 
`Department_Region` for inconsistent formatting (casing, whitespace, duplicate 
categories under different spellings). 

- Status: 3 clean values (Active, Pending, Inactive) — no issues found.
- Performance_Score:4 clean values (Average, Excellent, Good, Poor) — no issues 
found.
- Department_Region: 36 unique values, but this is expected — the column combines 
Department and Region (e.g., "DevOps-California"), so 36 reflects legitimate 
department×region combinations rather than inconsistent formatting.

No standardisation changes were necessary; all text fields were already consistently 
formatted.

In [39]:
summary = pd.DataFrame({
    'Metric': ['Total Rows', 'Duplicate Rows', 'Missing Values (Age)', 
               'Missing Values (Salary)', 'Join_Date dtype', 'Phone dtype'],
    'Before Cleaning': [1020, 0, 211, 24, 'object (str)', 'int64'],
    'After Cleaning': [df.shape[0], df.duplicated().sum(), 
                        df['Age'].isnull().sum(), df['Salary'].isnull().sum(),
                        str(df['Join_Date'].dtype), str(df['Phone'].dtype)]
})
summary

,Metric,Before Cleaning,After Cleaning
0,Total Rows,1020,1020
1,Duplicate Rows,0,0
2,Missing Values (Age),211,0
3,Missing Values (Salary),24,0
4,Join_Date dtype,object (str),datetime64[us]
5,Phone dtype,int64,str


In [ ]:
Before vs. After Summary:

The table above confirms all identified data quality 
issues were resolved: missing Age and Salary values eliminated, Join_Date correctly 
converted from text to datetime, and the corrupted Phone column safely converted to a 
labeled string type. Row count remained unchanged (1,020), confirming no data was lost 
during cleaning — only corrected or clearly flagged.

In [ ]:
## Conclusion

This dataset required cleaning across several dimensions: missing data (Age, Salary), 
incorrect data types (Join_Date, Phone), and verification checks for duplicates, 
outliers, and text standardisation.

Key decisions made:
1. Age's missing values were filled with the median, informed by the discovery that Age 
is limited to only 4 discrete buckets (25, 30, 35, 40) rather than continuous data.
2. Salary's missing values were filled with the median for consistency and robustness, 
though mean would have worked equally well given its low skew.
3. Phone numbers were found to be corrupted via integer overflow and could not be 
mathematically recovered — rather than fabricate replacement values, they were clearly 
labeled as corrupted to preserve transparency.
4. No duplicates, outliers, or text formatting inconsistencies were found, which was 
verified rather than assumed.

The cleaned dataset was saved as `cleaned_employee_dataset.csv`, ready for downstream 
analysis or modeling.

In [40]:
df.to_csv('cleaned_employee_dataset.csv', index=False)